In [3]:
"""
Comparação de Performance: 44 Features vs Features Selecionadas (MI).

Para um modelo específico, compara os sinais de trading gerados com
todos os 44 indicadores vs os gerados com features selecionadas por MI.

Calcula as 8 métricas de performance para cada ação e apresenta:
  - Média por setor
  - Média geral
  - Teste estatístico (Wilcoxon) para diferença significativa
  - Gráfico comparativo

Estrutura esperada:
  B3ICS/     → {Code}_TradeSignal_{ALGO}.csv  (44 features)
  B3ICSMI/   → {Code}_TradeSignal_{ALGO}_MI.csv (features selecionadas)

Uso:
    Edite ALGO abaixo e execute:
    python comparar_44_vs_MI.py
"""

from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import warnings

warnings.filterwarnings("ignore")

# ===================== CONFIGURAÇÃO =====================
# EDITE AQUI o modelo a comparar
ALGO = "DBN"

BASE_44 = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS")
BASE_MI = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICSMI")
SEC_NAMES = BASE_44 / ".NewB3_pruned.csv"
ANNUALIZATION = 250
# ========================================================


# ============================================================
#              MÉTRICAS DE PERFORMANCE
# ============================================================

def win_rate(r):
    nz = r[r != 0]
    return len(nz[nz > 0]) / len(nz) if len(nz) > 0 else 0.0

def ann_return(r):
    return r.mean() * ANNUALIZATION

def ann_sharpe(r):
    s = r.std(ddof=1)
    return np.sqrt(ANNUALIZATION) * r.mean() / s if s > 0 else 0.0

def max_drawdown(r):
    cum = (1 + r).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    return abs(dd.min()) if len(dd) > 0 else 0.0

def calmar(r):
    mdd = max_drawdown(r)
    return ann_return(r) / mdd if mdd > 0 else 0.0

def sortino(r, mar=0.0):
    down = r[r < mar]
    ds = np.sqrt((down ** 2).mean()) if len(down) > 0 else 0.0
    return (r.mean() - mar) / ds if ds > 0 else 0.0

def adj_sharpe(r):
    asr = ann_sharpe(r)
    s = r.std(ddof=1)
    if s == 0 or len(r) < 4: return asr
    m = r.mean()
    skew = ((r - m) ** 3).mean() / (s ** 3)
    kurt = ((r - m) ** 4).mean() / (s ** 4) - 3
    return asr * (1 + (skew / 6) * asr - (kurt / 24) * asr ** 2)

def avg_recovery(r):
    cum = (1 + r).cumprod()
    peak = cum.cummax()
    in_dd = cum < peak
    periods, length = [], 0
    for v in in_dd:
        if v: length += 1
        else:
            if length > 0: periods.append(length)
            length = 0
    if length > 0: periods.append(length)
    return np.mean(periods) if periods else 0.0

METRIC_FUNCS = {
    "WR": win_rate, "ARR": ann_return, "ASR": ann_sharpe,
    "MDD": max_drawdown, "CAR": calmar, "SOR": sortino,
    "MSR": adj_sharpe, "ART": avg_recovery,
}
METRIC_NAMES = list(METRIC_FUNCS.keys())


def compute_stock_metrics(signal_file, raw_file, start_date="2018-01-02", end_date="2019-12-31"):
    """Calcula as 8 métricas para uma ação a partir do arquivo de sinais."""
    if not signal_file.exists() or not raw_file.exists():
        return None

    try:
        sig_df = pd.read_csv(signal_file, encoding="utf-8-sig")
        sig_df[sig_df.columns[0]] = pd.to_datetime(sig_df[sig_df.columns[0]], errors="coerce")
        sig_df = sig_df.dropna(subset=[sig_df.columns[0]])
        signal = pd.Series(sig_df.iloc[:, 1].values,
                           index=pd.to_datetime(sig_df.iloc[:, 0].values))
        signal = signal.shift(1)
        signal.iloc[0] = 0

        raw_df = pd.read_csv(raw_file, encoding="utf-8-sig")
        raw_df[raw_df.columns[0]] = pd.to_datetime(raw_df[raw_df.columns[0]], errors="coerce")
        raw_df = raw_df.dropna(subset=[raw_df.columns[0]]).sort_values(raw_df.columns[0])
        raw_df = raw_df.set_index(raw_df.columns[0])
        raw_ret = raw_df["Close"].astype(float).pct_change()

        trade_ret = (signal * raw_ret).dropna()
        trade_ret = trade_ret.loc[start_date:end_date]

        if len(trade_ret) < 10:
            return None

        return {name: func(trade_ret) for name, func in METRIC_FUNCS.items()}
    except Exception:
        return None


def main():
    codes_df = pd.read_csv(SEC_NAMES, dtype=str, encoding="utf-8-sig")
    codes_df.columns = [c.strip() for c in codes_df.columns]
    codes_df["Code"] = codes_df["Code"].str.strip().str.upper()
    codes_df["industry"] = codes_df["industry"].str.strip()

    print("=" * 80)
    print(f"COMPARAÇÃO: {ALGO} com 44 Features vs {ALGO} com MI Features")
    print("=" * 80)

    results_44 = []
    results_mi = []

    for _, row in codes_df.iterrows():
        code = row["Code"]
        sector = row["industry"]

        raw_file = BASE_44 / f"{code}_New.csv"

        sig_44 = BASE_44 / f"{code}_TradeSignal_{ALGO}.csv"
        sig_mi = BASE_MI / f"{code}_TradeSignal_{ALGO}_MI.csv"

        met_44 = compute_stock_metrics(sig_44, raw_file)
        met_mi = compute_stock_metrics(sig_mi, raw_file)

        if met_44 is not None:
            met_44["Code"] = code
            met_44["Setor"] = sector
            results_44.append(met_44)

        if met_mi is not None:
            met_mi["Code"] = code
            met_mi["Setor"] = sector
            results_mi.append(met_mi)

    df_44 = pd.DataFrame(results_44)
    df_mi = pd.DataFrame(results_mi)

    print(f"\n  Ações com sinais (44 feat): {len(df_44)}")
    print(f"  Ações com sinais (MI feat): {len(df_mi)}")

    # Ações em comum
    common = set(df_44["Code"]) & set(df_mi["Code"])
    print(f"  Ações em comum: {len(common)}")

    df_44c = df_44[df_44["Code"].isin(common)].set_index("Code").sort_index()
    df_mic = df_mi[df_mi["Code"].isin(common)].set_index("Code").sort_index()

    # ============================================================
    # COMPARAÇÃO POR SETOR
    # ============================================================
    print(f"\n{'='*80}")
    print("MÉDIA POR SETOR")
    print(f"{'='*80}")

    setores = sorted(df_44c["Setor"].unique())

    for metric in METRIC_NAMES:
        print(f"\n  {metric}:")
        print(f"  {'Setor':>6} | {'44 feat':>10} | {'MI feat':>10} | {'Diff':>10} | {'Melhor':>10}")
        print(f"  {'-'*58}")

        for setor in setores:
            mask_44 = df_44c["Setor"] == setor
            mask_mi = df_mic["Setor"] == setor

            mean_44 = df_44c.loc[mask_44, metric].mean()
            mean_mi = df_mic.loc[mask_mi, metric].mean()
            diff = mean_mi - mean_44

            # Para MDD e ART, menor é melhor
            if metric in ["MDD", "ART"]:
                melhor = "MI" if mean_mi < mean_44 else "44"
            else:
                melhor = "MI" if mean_mi > mean_44 else "44"

            marker = "<<<" if melhor == "MI" else ""
            print(f"  {setor:>6} | {mean_44:>10.4f} | {mean_mi:>10.4f} | {diff:>+10.4f} | {melhor:>6} {marker}")

    # ============================================================
    # COMPARAÇÃO GERAL
    # ============================================================
    print(f"\n{'='*80}")
    print("MÉDIA GERAL (todas as ações)")
    print(f"{'='*80}")

    print(f"\n  {'Métrica':>8} | {'44 feat':>10} | {'MI feat':>10} | {'Diff':>10} | {'Melhor':>8} | {'p-valor':>10}")
    print(f"  {'-'*72}")

    summary = []
    for metric in METRIC_NAMES:
        vals_44 = df_44c[metric].values
        vals_mi = df_mic[metric].values

        mean_44 = vals_44.mean()
        mean_mi = vals_mi.mean()
        diff = mean_mi - mean_44

        # Wilcoxon signed-rank test (pareado)
        try:
            stat, pval = stats.wilcoxon(vals_mi, vals_44, alternative="two-sided")
        except Exception:
            pval = 1.0

        if metric in ["MDD", "ART"]:
            melhor = "MI" if mean_mi < mean_44 else "44"
        else:
            melhor = "MI" if mean_mi > mean_44 else "44"

        sig = "*" if pval < 0.05 else ("**" if pval < 0.01 else "")
        print(f"  {metric:>8} | {mean_44:>10.4f} | {mean_mi:>10.4f} | {diff:>+10.4f} | {melhor:>8} | {pval:>9.4f} {sig}")

        summary.append({
            "Metric": metric, "Mean_44": mean_44, "Mean_MI": mean_mi,
            "Diff": diff, "Better": melhor, "p_value": pval,
        })

    # ============================================================
    # SINAIS: comparação de distribuição
    # ============================================================
    print(f"\n{'='*80}")
    print("DISTRIBUIÇÃO DE SINAIS")
    print(f"{'='*80}")

    pct1_44_list = []
    pct1_mi_list = []

    for code in sorted(common):
        sig_44 = BASE_44 / f"{code}_TradeSignal_{ALGO}.csv"
        sig_mi = BASE_MI / f"{code}_TradeSignal_{ALGO}_MI.csv"

        if sig_44.exists():
            s44 = pd.read_csv(sig_44, encoding="utf-8-sig")
            pct1_44_list.append(s44.iloc[:, 1].mean() * 100)
        if sig_mi.exists():
            smi = pd.read_csv(sig_mi, encoding="utf-8-sig")
            pct1_mi_list.append(smi.iloc[:, 1].mean() * 100)

    print(f"  % sinais = 1 (média entre ações):")
    print(f"    44 features: {np.mean(pct1_44_list):.1f}%")
    print(f"    MI features: {np.mean(pct1_mi_list):.1f}%")

    # ============================================================
    # SALVAR RESULTADOS
    # ============================================================
    summary_df = pd.DataFrame(summary)
    outfile = BASE_44 / f"Comparacao_{ALGO}_44_vs_MI.csv"
    summary_df.to_csv(outfile, index=False, encoding="utf-8-sig")
    print(f"\nRelatório salvo: {outfile.name}")

    # Salvar detalhado por ação
    detail = df_44c[METRIC_NAMES].copy()
    detail.columns = [f"{m}_44" for m in METRIC_NAMES]
    for m in METRIC_NAMES:
        detail[f"{m}_MI"] = df_mic[m].values
        detail[f"{m}_diff"] = df_mic[m].values - df_44c[m].values
    detail["Setor"] = df_44c["Setor"].values
    detail_file = BASE_44 / f"Comparacao_{ALGO}_44_vs_MI_detalhe.csv"
    detail.to_csv(detail_file, encoding="utf-8-sig")
    print(f"Detalhe por ação: {detail_file.name}")

    # ============================================================
    # GRÁFICO
    # ============================================================
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(2, 4, figsize=(18, 8))
        fig.suptitle(f"Comparação {ALGO}: 44 Features vs MI Features", fontsize=14)

        for idx, metric in enumerate(METRIC_NAMES):
            ax = axes[idx // 4, idx % 4]

            vals_44 = df_44c[metric].values
            vals_mi = df_mic[metric].values

            bp = ax.boxplot([vals_44, vals_mi], labels=["44 feat", "MI feat"],
                           patch_artist=True)
            bp["boxes"][0].set_facecolor("lightblue")
            bp["boxes"][1].set_facecolor("lightyellow")

            ax.set_title(metric, fontweight="bold")
            ax.grid(True, alpha=0.3)

        fig.tight_layout()
        plot_file = BASE_44 / f"Comparacao_{ALGO}_44_vs_MI.png"
        fig.savefig(plot_file, dpi=150)
        plt.close(fig)
        print(f"Gráfico salvo: {plot_file.name}")
    except ImportError:
        pass


if __name__ == "__main__":
    main()

COMPARAÇÃO: DBN com 44 Features vs DBN com MI Features

  Ações com sinais (44 feat): 239
  Ações com sinais (MI feat): 239
  Ações em comum: 239

MÉDIA POR SETOR

  WR:
   Setor |    44 feat |    MI feat |       Diff |     Melhor
  ----------------------------------------------------------
      BM |     0.4357 |     0.4433 |    +0.0077 |     MI <<<
      CC |     0.4518 |     0.4717 |    +0.0200 |     MI <<<
     CNC |     0.4087 |     0.4463 |    +0.0376 |     MI <<<
     COM |     0.2283 |     0.4386 |    +0.2104 |     MI <<<
     FIN |     0.4833 |     0.4589 |    -0.0244 |     44 
     GCS |     0.4725 |     0.4606 |    -0.0120 |     44 
     OGB |     0.5409 |     0.4330 |    -0.1079 |     44 
     UTI |     0.4473 |     0.4498 |    +0.0025 |     MI <<<

  ARR:
   Setor |    44 feat |    MI feat |       Diff |     Melhor
  ----------------------------------------------------------
      BM |     0.1703 |     0.1960 |    +0.0257 |     MI <<<
      CC |     0.2564 |     0.2716 |  